# Continuous Simulation & The Lorenz Attractor

Welcome to the first interactive notebook of the `digital-twins` repository. 

Continuous simulation models are defined by Ordinary Differential Equations (ODEs). The state changes continuously over time. To simulate this on a digital computer, we must discretize time into steps ($\Delta t$) and use a **Numerical Integrator** (like Euler or Runge-Kutta) to approximate the continuous curve.

### The Lorenz System  
The Lorenz system is a classic continuous model known for its chaotic behavior (the "Butterfly Effect"). It is defined by three coupled ODEs:

$$ \frac{dx}{dt} = \sigma(y - x) $$
$$ \frac{dy}{dt} = x(\rho - z) - y $$
$$ \frac{dz}{dt} = xy - \beta z $$

**Why this matters for Digital Twins:**
In chaotic systems, an infinitely small error in our initial state estimate ($x_0$) will cause our simulation to diverge entirely from reality over time. This mathematically proves why offline simulation is not enough for real-time decision making, and why we *must* continuously assimilate live sensor data to correct our DT.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

# Ensure Python can find the 'src' directory in the root of the repository

from digital_twins.models.continuous import LorenzSystem, ContinuousSimulator

### Interactive Learning: Numerical Instability & Chaos

In the cell below, we simulate the Lorenz system. You have three controls:

1. **Integrator Method (`euler` vs `rk4`)**: While simple, Euler is highly inaccurate for complex curves. RK4 (Runge-Kutta 4th Order) is much more stable.
2. **Time Step `dt` ($\Delta t$)**: Try increasing `dt` using the `euler` method. Watch how the simulation visually "blows up" and becomes numerically unstable. Then switch to `rk4` and see how it handles the exact same `dt` flawlessly.
3. **Initial Perturbation (Chaos)**: We run a second, "Perturbed" simulation (red dotted line) alongside the base simulation (blue line). The perturbed simulation starts with an initial $X$ value that is shifted by a microscopic fraction. **Gradually increase the perturbation slider to `0.001` and watch how quickly the red line completely deviates from the blue line.**

In [ ]:
def interactive_lorenz(method='rk4', dt=0.01, perturbation=0.0):
    # 1. Initialize the Model and Simulator from our source code
    lorenz_model = LorenzSystem(sigma=10.0, rho=28.0, beta=8.0/3.0)
    simulator = ContinuousSimulator(model=lorenz_model, method=method)
    
    # 2. Define Initial States
    x0_base = np.array([1.0, 1.0, 1.0])
    x0_pert = np.array([1.0 + perturbation, 1.0, 1.0])
    
    # 3. Run Simulations
    try:
        t_hist, x_base, _ = simulator.simulate(t_start=0.0, t_end=30.0, dt=dt, x0=x0_base)
        if perturbation > 0:
            _, x_pert, _ = simulator.simulate(t_start=0.0, t_end=30.0, dt=dt, x0=x0_pert)
    except OverflowError:
        print("Simulation Failed: Numerical Instability! (dt is likely too high for the chosen method)")
        return
    
    # 4. Plotting (Recreating Textbook Figure 3.5)
    fig = plt.figure(figsize=(14, 6))
    
    # Plot A: Time Series (X coordinate over time)
    ax1 = fig.add_subplot(121)
    ax1.plot(t_hist, x_base[:, 0], label='Base Trajectory', color='royalblue', linewidth=1.5)
    if perturbation > 0:
        ax1.plot(t_hist, x_pert[:, 0], label='Perturbed Trajectory', color='crimson', linestyle='--', linewidth=1.5)
        
    ax1.set_title("Time Series: X-Coordinate over Time", fontweight='bold')
    ax1.set_xlabel("Time (t)")
    ax1.set_ylabel("X value")
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot B: Phase Plane (X vs Z)
    ax2 = fig.add_subplot(122)
    ax2.plot(x_base[:, 0], x_base[:, 2], label='Base', color='royalblue', alpha=0.7, linewidth=1.0)
    if perturbation > 0:
        ax2.plot(x_pert[:, 0], x_pert[:, 2], label='Perturbed', color='crimson', linestyle='--', alpha=0.7, linewidth=1.0)
        
    ax2.set_title("Phase Plane: X-Z Trajectory (The Attractor)", fontweight='bold')
    ax2.set_xlabel("X position")
    ax2.set_ylabel("Z position")
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Create the interactive widgets
interact(interactive_lorenz, 
         method=Dropdown(options=['rk4', 'euler'], value='rk4', description='Integrator:'),
         dt=FloatSlider(value=0.01, min=0.005, max=0.05, step=0.005, description='dt size:', continuous_update=False),
         perturbation=FloatSlider(value=0.0, min=0.0, max=0.005, step=0.0001, readout_format='.4f', description='Add Error x0:', continuous_update=False))